In [1]:
import pandas as pd

print(pd.__version__)

3.0.3


In [2]:
# Import required libraries

import pandas as pd
import re
from datasets import load_dataset

In [4]:
# Load CNN/DailyMail dataset
# Purpose:
# - Load dataset for EDA


cnn_dataset = load_dataset(
    "cnn_dailymail",
    "3.0.0",
    split="train[:20003]"
)

cnn_dataset

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 20003
})

In [5]:
# Convert Hugging Face dataset into DataFrame
# Purpose:
# - Enable Pandas-based analysis

cnn_dataset = cnn_dataset.to_pandas()

cnn_dataset.head()

,article,highlights,id
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...,42c027e4ff9730fbb3de84c1af0d2c506e41c3e4
1,Editor's note: In our Behind the Scenes series...,Mentally ill inmates in Miami are housed on th...,ee8871b15c50d0db17b0179a6d2beab35065f1e9
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...","NEW: ""I thought I was going to die,"" driver sa...",06352019a19ae31e527f37f7571c6dd7f0c5da37
3,WASHINGTON (CNN) -- Doctors removed five small...,"Five small polyps found during procedure; ""non...",24521a2abb2e1f5e34e6824e0f9e56904a2b0e88
4,(CNN) -- The National Football League has ind...,"NEW: NFL chief, Atlanta Falcons owner critical...",7fe70cc8b12fab2d0a258fababf7d9c6b5e1262a


In [6]:
# Dataset dimensions
# Purpose:
# - Understand total records and features

cnn_dataset.shape

(20003, 3)

In [7]:
# View dataset columns
# Purpose:
# - Understand available features

cnn_dataset.columns

Index(['article', 'highlights', 'id'], dtype='str')

In [8]:
# Dataset information
# Purpose:
# - Inspect data types
# - Verify completeness

cnn_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 20003 entries, 0 to 20002
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   article     20003 non-null  str  
 1   highlights  20003 non-null  str  
 2   id          20003 non-null  str  
dtypes: str(3)
memory usage: 74.1 MB


The CNN/DailyMail sample dataset contains 100 complete records with no missing values.

The dataset consists of:
- article     : Original news article
- highlights  : Human-written summary
- id          : Unique article identifier

Unlike the BBC and Inshorts datasets, CNN/DailyMail provides both source articles and reference summaries, making it highly suitable for summarization model evaluation.

In [9]:
# Missing value analysis
# Purpose:
# - Identify incomplete records

cnn_dataset.isnull().sum()

article       0
highlights    0
id            0
dtype: int64

In [10]:
# Duplicate analysis
# Purpose:
# - Prevent duplicate embeddings

cnn_dataset.duplicated().sum()

np.int64(0)

In [11]:
# Inspect sample article and summary
# Purpose:
# - Understand dataset structure

cnn_dataset.loc[0]

article       LONDON, England (Reuters) -- Harry Potter star...
highlights    Harry Potter star Daniel Radcliffe gets £20M f...
id                     42c027e4ff9730fbb3de84c1af0d2c506e41c3e4
Name: 0, dtype: str

In [12]:
# Article length analysis
# Purpose:
# - Understand article size
# - Support chunking decisions

cnn_dataset["article_word_count"] = (
    cnn_dataset["article"]
    .astype(str)
    .apply(lambda x: len(x.split()))
)

cnn_dataset["article_word_count"].describe()

count    20003.000000
mean       596.814828
std        319.067571
min         18.000000
25%        348.000000
50%        533.000000
75%        792.000000
max       1831.000000
Name: article_word_count, dtype: float64

In [13]:
# Summary length analysis
# Purpose:
# - Understand target summary size

cnn_dataset["summary_word_count"] = (
    cnn_dataset["highlights"]
    .astype(str)
    .apply(lambda x: len(x.split()))
)

cnn_dataset["summary_word_count"].describe()

count    20003.000000
mean        44.434135
std          8.977947
min         11.000000
25%         38.000000
50%         45.000000
75%         51.000000
max         90.000000
Name: summary_word_count, dtype: float64

In [14]:
# Compression ratio analysis
# Purpose:
# - Compare article and summary lengths
# - Understand summarization compression

cnn_dataset["compression_ratio"] = (
    cnn_dataset["summary_word_count"]
    /
    cnn_dataset["article_word_count"]
)

cnn_dataset["compression_ratio"].describe()

count    20003.000000
mean         0.098095
std          0.062003
min          0.014002
25%          0.056857
50%          0.083110
75%          0.124130
max          2.111111
Name: compression_ratio, dtype: float64

In [15]:
# Character length analysis
# Purpose:
# - Estimate document size
# - Support chunking and embedding decisions

cnn_dataset["character_count"] = (
    cnn_dataset["article"]
    .astype(str)
    .apply(len)
)

cnn_dataset["character_count"].describe()

count    20003.000000
mean      3561.497225
std       1877.673616
min        106.000000
25%       2093.000000
50%       3187.000000
75%       4717.000000
max      11441.000000
Name: character_count, dtype: float64

In [16]:
# Calculate vocabulary size
# Purpose:
# - Measure language diversity
# - Understand dataset richness

all_text = " ".join(
    cnn_dataset["article"]
    .astype(str)
)

vocabulary = set(
    all_text.lower().split()
)

print(
    "Vocabulary Size:",
    len(vocabulary)
)

Vocabulary Size: 307906


In [17]:
# Segment article into sentences
# Purpose:
# - Understand article structure
# - Support chunking strategy

import re

sample_article = cnn_dataset.loc[
    0,
    "article"
]

sentences = re.split(
    r'(?<=[.!?])\s+',
    sample_article
)

print(
    "Number of Sentences:",
    len(sentences)
)

sentences[:5]

Number of Sentences: 20


["LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him.",
 'Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties.',
 '"I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month.',
 '"I don\'t think I\'ll be particularly extravagant.',
 '"The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office 

In [18]:
# Calculate sentence count for all articles
# Purpose:
# - Measure article complexity
# - Support chunking strategy

cnn_dataset["sentence_count"] = (
    cnn_dataset["article"]
    .astype(str)
    .apply(
        lambda x: len(
            re.split(
                r'(?<=[.!?])\s+',
                x
            )
        )
    )
)

cnn_dataset["sentence_count"].describe()

count    20003.000000
mean        30.132830
std         19.663721
min          1.000000
25%         16.000000
50%         25.000000
75%         39.000000
max        173.000000
Name: sentence_count, dtype: float64

In [19]:
# Compare article and reference summary lengths
# Purpose:
# - Understand summarization compression

sample_summary = cnn_dataset.loc[
    0,
    "highlights"
]

article_words = len(
    sample_article.split()
)

summary_words = len(
    sample_summary.split()
)

print(
    "Article Words:",
    article_words
)

print(
    "Summary Words:",
    summary_words
)

print(
    "Compression Ratio:",
    round(
        summary_words /
        article_words,
        2
    )
)

Article Words: 455
Summary Words: 41
Compression Ratio: 0.09


In [20]:
cnn_dataset["article_word_count"].describe()

cnn_dataset["summary_word_count"].describe()

cnn_dataset["compression_ratio"].describe()

cnn_dataset["character_count"].describe()

len(vocabulary)

cnn_dataset["sentence_count"].describe()

count    20003.000000
mean        30.132830
std         19.663721
min          1.000000
25%         16.000000
50%         25.000000
75%         39.000000
max        173.000000
Name: sentence_count, dtype: float64

CNN/DailyMail articles are significantly longer than Inshorts articles.

On average, the reference summary contains only about 9% of the original article content.

This indicates a high compression ratio and makes the dataset highly suitable for evaluating summarization models.

### PRE-Processing


In [21]:
# Remove missing values
# Purpose:
# - Ensure all articles and summaries are usable

cnn_dataset = (
    cnn_dataset.dropna()
)

cnn_dataset.isnull().sum()

article               0
highlights            0
id                    0
article_word_count    0
summary_word_count    0
compression_ratio     0
character_count       0
sentence_count        0
dtype: int64

In [22]:
# Remove duplicate records
# Purpose:
# - Prevent duplicate embeddings
# - Improve retrieval quality

print(
    "Duplicates Before:",
    cnn_dataset.duplicated().sum()
)

cnn_dataset = (
    cnn_dataset.drop_duplicates()
)

print(
    "Duplicates After:",
    cnn_dataset.duplicated().sum()
)

Duplicates Before: 0
Duplicates After: 0


In [23]:
# Create text cleaning function
# Purpose:
# - Standardize text
# - Remove unwanted characters
# - Prepare for NLP processing

def clean_text(text):

    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove special characters
    text = re.sub(
        r'[^a-zA-Z0-9\s]',
        '',
        text
    )

    # Remove extra spaces
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text

In [24]:
# Clean article content
# Purpose:
# - Create cleaned article text

cnn_dataset["clean_article"] = (
    cnn_dataset["article"]
    .apply(clean_text)
)

In [25]:
# Clean reference summaries
# Purpose:
# - Standardize summary text

cnn_dataset["clean_summary"] = (
    cnn_dataset["highlights"]
    .apply(clean_text)
)

In [26]:
# Tokenize all articles
# Purpose:
# - Split articles into words
# - Prepare for NLP processing

cnn_dataset["tokens"] = (
    cnn_dataset["clean_article"]
    .apply(lambda x: x.split())
)

In [27]:
# Define stopwords
# Purpose:
# - Remove low-information words

stop_words = {
    "a", "an", "the", "is", "are",
    "was", "were", "to", "of",
    "in", "on", "for", "and",
    "with", "at", "from", "by",
    "as", "that", "this", "it"
}

In [28]:
# Remove stopwords from all articles
# Purpose:
# - Improve retrieval quality
# - Retain meaningful words

cnn_dataset["filtered_tokens"] = (
    cnn_dataset["tokens"]
    .apply(
        lambda tokens: [
            word
            for word in tokens
            if word not in stop_words
        ]
    )
)

In [29]:
# Create NLP-ready text
# Purpose:
# - Prepare text for chunking and embeddings

cnn_dataset["nlp_content"] = (
    cnn_dataset["filtered_tokens"]
    .apply(
        lambda tokens: " ".join(tokens)
    )
)

In [30]:
# Compare original and processed article
# Purpose:
# - Verify preprocessing quality

print("ORIGINAL ARTICLE:\n")

print(
    cnn_dataset.loc[
        0,
        "article"
    ][:500]
)

print("\n" + "="*100 + "\n")

print("NLP READY ARTICLE:\n")

print(
    cnn_dataset.loc[
        0,
        "nlp_content"
    ][:500]
)

ORIGINAL ARTICLE:

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s


NLP READY ARTICLE:

london england reuters harry potter star daniel radcliffe gains access reported 20 million 411 million fortune he turns 18 monday but he insists money wont cast spell him daniel radcliffe harry potter harry potter order phoenix disappointment gossip columnists around world young actor says he has no plans fritter his cash away fast cars drink celebrity parties i dont plan be one those people who soon they turn 18 suddenly buy themselves massive sports ca

In [31]:
# Final dataset verification
# Purpose:
# - Confirm successful preprocessing and NLP

print(
    "Total Articles:",
    len(cnn_dataset)
)

print(
    "Columns:"
)

print(
    list(cnn_dataset.columns)
)

Total Articles: 20003
Columns:
['article', 'highlights', 'id', 'article_word_count', 'summary_word_count', 'compression_ratio', 'character_count', 'sentence_count', 'clean_article', 'clean_summary', 'tokens', 'filtered_tokens', 'nlp_content']


In [32]:
print(len(cnn_dataset))
print(cnn_dataset.columns)

20003
Index(['article', 'highlights', 'id', 'article_word_count',
       'summary_word_count', 'compression_ratio', 'character_count',
       'sentence_count', 'clean_article', 'clean_summary', 'tokens',
       'filtered_tokens', 'nlp_content'],
      dtype='str')


In [33]:
cnn_dataset.columns

Index(['article', 'highlights', 'id', 'article_word_count',
       'summary_word_count', 'compression_ratio', 'character_count',
       'sentence_count', 'clean_article', 'clean_summary', 'tokens',
       'filtered_tokens', 'nlp_content'],
      dtype='str')

In [34]:
cnn_dataset.to_csv(
    "cnn_processed20k.csv",
    index=False
)